# Actividad Académica: Aprendizaje Supervisado con Árboles y Reglas de Decisión
## Caso de Estudio: Sistema de Transporte Masivo (TransMilenio / Metro / SITP)

**Docente / Asignatura:** Inteligencia Artificial & Machine Learning  
**Tema:** Clasificación mediante Árboles de Decisión, Extracción de Reglas de Inferencia y Evaluación de Modelos.  

---

### Objetivos de la Actividad:
1. **Generar un dataset representativo** con variables de tráfico, ambientales, operacionales y estado del servicio.
2. **Implementar un flujo de preprocesamiento estructurado** usando `ColumnTransformer` y `Pipeline` de Scikit-Learn.
3. **Entrenar un modelo de Árbol de Decisión (`DecisionTreeClassifier`)** con hiperparámetros controlados para evitar *overfitting*.
4. **Extraer el conjunto de Reglas de Decisión Si-Entonces (IF-THEN)** en formato texto para su interpretación formal.
5. **Evaluar el rendimiento del clasificador** a través de matrices de confusión, curvas e indicadores (Precision, Recall, F1-Score).
6. **Validar inferencia en tiempo real** simulando condiciones imprevistas de congestión e incidentes viales.

## 1. Importación de Librerías y Configuración de Entorno

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Estética de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

## 2. Generación del Dataset Sintético

Se generan 1,500 observaciones simulando operaciones de transporte masivo con variables correlacionadas de acuerdo con la dinámica de tráfico real.

In [ ]:
def generar_dataset_transporte(n_samples: int = 1500, random_state: int = 42) -> pd.DataFrame:
    np.random.seed(random_state)

    # 1. Variables Temporales y Clima
    horas = np.random.randint(5, 24, size=n_samples)
    dias = np.random.choice(
        ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo'],
        size=n_samples,
        p=[0.16, 0.16, 0.16, 0.16, 0.16, 0.11, 0.09]
    )

    # Hora Pico: Lunes a Viernes (6:00-8:59 y 17:00-19:59)
    es_hora_pico = [
        1 if (d in ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes']) and ((6 <= h <= 8) or (17 <= h <= 19))
        else 0
        for h, d in zip(horas, dias)
    ]
    es_hora_pico = np.array(es_hora_pico)

    climas = np.random.choice(
        ['Despejado', 'Lluvia ligera', 'Lluvia fuerte'],
        size=n_samples,
        p=[0.60, 0.28, 0.12]
    )

    # 2. Infraestructura y Operación
    estaciones = np.random.choice(
        ['Portal Norte', 'Calle 100', 'Portal Américas', 'Ricaurte', 'Portal El Dorado', 'Banderas', 'Héroes'],
        size=n_samples,
        p=[0.18, 0.18, 0.18, 0.16, 0.12, 0.10, 0.08]
    )

    base_afluencia = np.where(es_hora_pico == 1, np.random.normal(2500, 400, n_samples), np.random.normal(900, 250, n_samples))
    factor_estacion = np.isin(estaciones, ['Portal Américas', 'Portal Norte', 'Calle 100']) * 300
    afluencia_pasajeros = np.clip(base_afluencia + factor_estacion, 150, 4000).astype(int)

    frecuencia_buses_min = np.where(
        es_hora_pico == 1,
        np.random.choice([3, 4, 5, 6], size=n_samples, p=[0.4, 0.35, 0.15, 0.1]),
        np.random.choice([6, 8, 10, 12, 15], size=n_samples, p=[0.2, 0.3, 0.25, 0.15, 0.1])
    )

    prob_incidente = np.where(
        climas == 'Lluvia fuerte', 0.25,
        np.where(es_hora_pico == 1, 0.12, 0.04)
    )
    incidente_vial = (np.random.rand(n_samples) < prob_incidente).astype(int)

    # 3. Modelado Causal del Target (nivel_retraso)
    puntaje_retraso = (
        (es_hora_pico * 2.8) +
        (incidente_vial * 4.5) +
        (climas == 'Lluvia fuerte') * 3.2 +
        (climas == 'Lluvia ligera') * 1.1 +
        (afluencia_pasajeros / 1000.0 * 1.0) +
        (frecuencia_buses_min * 0.15) +
        np.random.normal(0, 1.0, size=n_samples)
    )

    niveles = []
    for p in puntaje_retraso:
        if p < 4.2:
            niveles.append('Normal')
        elif p < 7.5:
            niveles.append('Retraso_Moderado')
        else:
            niveles.append('Retraso_Critico')

    df = pd.DataFrame({
        'hora_del_dia': horas,
        'dia_semana': dias,
        'es_hora_pico': es_hora_pico,
        'clima': climas,
        'estacion_origen': estaciones,
        'afluencia_pasajeros': afluencia_pasajeros,
        'frecuencia_buses_min': frecuencia_buses_min,
        'incidente_vial': incidente_vial,
        'nivel_retraso': niveles
    })
    return df

# Generar y exportar
df = generar_dataset_transporte(n_samples=1500, random_state=42)
df.to_csv('dataset_transporte_masivo.csv', index=False, encoding='utf-8')
print(f"Dataset generado con {df.shape[0]} registros y guardado en 'dataset_transporte_masivo.csv'.")

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Distribución de Clases del Target
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='nivel_retraso', palette='Blues_r', order=['Normal', 'Retraso_Moderado', 'Retraso_Critico'])
plt.title('Distribución de Clases (Nivel de Retraso)', fontsize=13, weight='bold')
plt.xlabel('Nivel de Retraso')
plt.ylabel('Frecuencia')
plt.show()

## 4. Pipeline de Preprocesamiento y Entrenamiento

Utilizamos `ColumnTransformer` para estandarizar variables numéricas y codificar variables categóricas con `OneHotEncoder`. Posteriormente, dividimos el conjunto de datos con estratificación (80/20) y entrenamos el `DecisionTreeClassifier`.

In [ ]:
X = df.drop(columns=['nivel_retraso'])
y = df['nivel_retraso']

features_numericas = ['hora_del_dia', 'afluencia_pasajeros', 'frecuencia_buses_min']
features_categoricas = ['dia_semana', 'clima', 'estacion_origen', 'es_hora_pico', 'incidente_vial']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_categoricas)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

tree_clf = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('clasificador', tree_clf)
])

pipeline.fit(X_train, y_train)
print("Modelo entrenado exitosamente.")

## 5. Extracción de Reglas de Decisión y Visualización del Árbol

In [ ]:
# Obtener nombres de las columnas transformadas
ohe_step = pipeline.named_steps['preprocesamiento'].named_transformers_['cat']
cat_feature_names = ohe_step.get_feature_names_out(features_categoricas)
all_feature_names = features_numericas + list(cat_feature_names)
trained_tree = pipeline.named_steps['clasificador']
classes = [str(c) for c in trained_tree.classes_]

# Extracción de Reglas de Decisión en Texto
reglas = export_text(trained_tree, feature_names=all_feature_names, spacing=3, decimals=2)
print("=== REGLAS DE DECISIÓN FORMALES ===\n")
print(reglas)

In [ ]:
# Visualización del Árbol de Decisión
plt.figure(figsize=(22, 10), dpi=200)
plot_tree(
    trained_tree,
    feature_names=all_feature_names,
    class_names=classes,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Estructura del Árbol de Decisión - Retrasos en Transporte', fontsize=16, weight='bold', pad=15)
plt.show()

In [ ]:
# Importancia de las Características
importancias = trained_tree.feature_importances_
df_imp = pd.DataFrame({'Característica': all_feature_names, 'Importancia': importancias})
df_imp = df_imp[df_imp['Importancia'] > 0].sort_values(by='Importancia', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(df_imp['Característica'], df_imp['Importancia'], color='#2b5c8f', edgecolor='black')
plt.xlabel('Importancia de Gini', weight='bold')
plt.title('Importancia Relativa de Variables Predictoras', weight='bold', fontsize=13)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 6. Evaluación del Desempeño en Test

In [ ]:
y_pred = pipeline.predict(X_test)
print("=== REPORTE DE CLASIFICACIÓN ===\n")
print(classification_report(y_test, y_pred, digits=4))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusión (Conjunto de Prueba)', weight='bold')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

## 7. Inferencia con Nuevas Observaciones Operacionales

In [ ]:
nuevos_datos = pd.DataFrame([
    {
        'hora_del_dia': 18,
        'dia_semana': 'Miércoles',
        'es_hora_pico': 1,
        'clima': 'Lluvia fuerte',
        'estacion_origen': 'Portal Américas',
        'afluencia_pasajeros': 3400,
        'frecuencia_buses_min': 4,
        'incidente_vial': 1
    },
    {
        'hora_del_dia': 11,
        'dia_semana': 'Martes',
        'es_hora_pico': 0,
        'clima': 'Despejado',
        'estacion_origen': 'Héroes',
        'afluencia_pasajeros': 650,
        'frecuencia_buses_min': 8,
        'incidente_vial': 0
    }
])

preds = pipeline.predict(nuevos_datos)
probs = pipeline.predict_proba(nuevos_datos)

for i, p in enumerate(preds):
    print(f"Escenario {i+1} -> Predicción: {p} | Probabilidades: {dict(zip(classes, np.round(probs[i], 3)))}")